In [19]:
import os
import glob
import pandas as pd

df = pd.read_csv("/home/arch/Downloads/Alert-grouping-pipeline/output/russellmitchell_alerts_grouped.csv")

df

,time,name,short,short_abbreviation,agent_ip,host,rule_id,level,decoder,location,...,src_port,dst_ip,dst_port,protocol,alert_id,ip,readable_time,ip_subnet,timely_alert_group,initial_time_group
0,1642723201,AMiner,AMiner: New event type.,A-All-Evt,172.19.130.106,cloud_share,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,172.19.130.106,172.19.130.106,2022-01-21 00:00:01,172.19.130,1,1
1,1642723201,AMiner,AMiner: New service_stop parameter combination...,A-Aud-Com4,172.19.130.106,cloud_share,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,172.19.130.106,172.19.130.106,2022-01-21 00:00:01,172.19.130,1,1
2,1642723201,AMiner,AMiner: New login parameter combination in Aud...,A-Aud-Com3,172.19.130.106,cloud_share,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,172.19.130.106,172.19.130.106,2022-01-21 00:00:01,172.19.130,1,1
3,1642723201,AMiner,AMiner: New cred_refr parameter combination in...,A-Aud-Com2,172.19.130.106,cloud_share,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,172.19.130.106,172.19.130.106,2022-01-21 00:00:01,172.19.130,1,1
4,1642723201,AMiner,AMiner: New user_start parameter combination i...,A-Aud-Com6,172.19.130.106,cloud_share,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,172.19.130.106,172.19.130.106,2022-01-21 00:00:01,172.19.130,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45539,1642996637,AMiner,AMiner: New event type.,A-All-Evt,192.168.231.56,davey_mail,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,192.168.231.56,192.168.231.56,2022-01-24 03:57:17,192.168.231,5536,3046
45540,1642996637,AMiner,AMiner: New event type.,A-All-Evt,192.168.231.56,davey_mail,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,192.168.231.56,192.168.231.56,2022-01-24 03:57:17,192.168.231,5536,3046
45541,1642996637,AMiner,AMiner: New event type.,A-All-Evt,192.168.231.56,davey_mail,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,192.168.231.56,192.168.231.56,2022-01-24 03:57:17,192.168.231,5536,3046
45542,1642996637,AMiner,AMiner: New event type.,A-All-Evt,192.168.231.56,davey_mail,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,192.168.231.56,192.168.231.56,2022-01-24 03:57:17,192.168.231,5536,3046


In [20]:
df.columns

Index(['time', 'name', 'short', 'short_abbreviation', 'agent_ip', 'host',
       'rule_id', 'level', 'decoder', 'location', 'src_ip', 'src_port',
       'dst_ip', 'dst_port', 'protocol', 'alert_id', 'ip', 'readable_time',
       'ip_subnet', 'timely_alert_group', 'initial_time_group'],
      dtype='object')

In [21]:
import os
import glob
import pandas as pd


OUTPUT_DIR = "/home/arch/Downloads/Alert-grouping-pipeline/output"

IDS_NAMES = ["Suricata", "Wazuh", "AMiner"]


def calculate_grouped_alert_statistics(output_dir=OUTPUT_DIR):
    """
    Calculate alert-grouping statistics from all *_alerts_grouped.csv files.

    Expected columns:
        name
        timely_alert_group

    Returns
    -------
    pandas.DataFrame
        Columns:
            Dataset
            Level
            Name
            Alerts Before Groups
            Groups After
            Reduction
            Alerts Not Assigned to Any Cluster
            Coverage
    """

    files = glob.glob(
        os.path.join(output_dir, "*_alerts_grouped.csv")
    )

    results = []

    for filepath in files:

        filename = os.path.basename(filepath)

        # Remove the suffix
        scenario = filename.replace("_alerts_grouped.csv", "")

        # ---------------------------------------------------------
        # Adjust this if your filename format is different.
        #
        # Example:
        #     dataset_level_alerts_grouped.csv
        #
        # gives:
        #     Dataset = dataset
        #     Level   = level
        # ---------------------------------------------------------
        parts = scenario.split("_")

        dataset = parts[0]

        if len(parts) > 1:
            level = "_".join(parts[1:])
        else:
            level = ""

        df = pd.read_csv(filepath)

        required_columns = {"name", "timely_alert_group"}

        missing = required_columns - set(df.columns)

        if missing:
            print(
                f"Skipping {filename}: "
                f"missing columns {sorted(missing)}"
            )
            continue

        # Keep only the three IDS types we care about
        df = df[df["name"].isin(IDS_NAMES)].copy()

        # ---------------------------------------------------------
        # Statistics for each IDS
        # ---------------------------------------------------------
        for name in IDS_NAMES:

            ids_df = df[df["name"] == name].copy()

            alerts_before = len(ids_df)

            if alerts_before == 0:
                groups_after = 0
                reduction = 0.0
                alerts_not_assigned = 0
                coverage = 0.0

            else:
                # Each unique timely_alert_group represents one
                # inspection unit after grouping.
                groups_after = ids_df["timely_alert_group"].nunique()

                reduction = (
                    1 - (groups_after / alerts_before)
                )

                # -------------------------------------------------
                # Coverage
                #
                # An alert is not assigned to a cluster when its
                # timely_alert_group occurs only once.
                #
                # Example:
                #   group A -> 3 alerts
                #   group B -> 2 alerts
                #   group C -> 1 alert
                #
                # R = 1
                # N = 6
                # Coverage = (6 - 1) / 6
                #           = 0.8333
                # -------------------------------------------------
                group_sizes = (
                    ids_df["timely_alert_group"]
                    .value_counts()
                )

                alerts_not_assigned = (
                    group_sizes[group_sizes == 1].sum()
                )

                coverage = (
                    (alerts_before - alerts_not_assigned)
                    / alerts_before
                )

            results.append({
                "Dataset": dataset,
                "Level": level,
                "Name": name,
                "Alerts Before Groups": alerts_before,
                "Groups After": groups_after,
                "Reduction": reduction,
                "Alerts Not Assigned to Any Cluster": alerts_not_assigned,
                "Coverage": coverage
            })

        # ---------------------------------------------------------
        # All IDSs
        # ---------------------------------------------------------
        alerts_before = len(df)

        if alerts_before == 0:
            groups_after = 0
            reduction = 0.0
            alerts_not_assigned = 0
            coverage = 0.0

        else:
            # IMPORTANT:
            # Count unique groups across all IDSs.
            #
            # This means that if the same timely_alert_group ID
            # occurs for multiple IDSs, it is counted once.
            groups_after = df["timely_alert_group"].nunique()

            reduction = (
                1 - (groups_after / alerts_before)
            )

            # -----------------------------------------------------
            # Coverage across all IDSs
            #
            # Group sizes are calculated across the complete
            # combined dataframe.
            #
            # Therefore, if the same timely_alert_group appears
            # for multiple IDSs, it is treated as one group and
            # is NOT considered a singleton.
            # -----------------------------------------------------
            group_sizes = (
                df["timely_alert_group"]
                .value_counts()
            )

            alerts_not_assigned = (
                group_sizes[group_sizes == 1].sum()
            )

            coverage = (
                (alerts_before - alerts_not_assigned)
                / alerts_before
            )

        results.append({
            "Dataset": dataset,
            "Level": level,
            "Name": "All IDSs",
            "Alerts Before Groups": alerts_before,
            "Groups After": groups_after,
            "Reduction": reduction,
            "Alerts Not Assigned to Any Cluster": alerts_not_assigned,
            "Coverage": coverage
        })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df["Reduction"] = (
            results_df["Reduction"] * 100
        )

        results_df["Coverage"] = (
            results_df["Coverage"] * 100
        )

    return results_df



In [22]:
statistics = calculate_grouped_alert_statistics()

print(statistics.to_string(index=False))

/tmp/ipykernel_6341/2134063648.py:65: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)
/tmp/ipykernel_6341/2134063648.py:65: DtypeWarning: Columns (12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


        Dataset Level     Name  Alerts Before Groups  Groups After  Reduction  Alerts Not Assigned to Any Cluster  Coverage
       wardbeck       Suricata                 24510          4487  81.693186                                  16 99.934721
       wardbeck          Wazuh                 63694         11463  82.003014                                 357 99.439508
       wardbeck         AMiner                  3053           552  81.919424                                 219 92.826728
       wardbeck       All IDSs                 91257         11812  87.056335                                 498 99.454288
         santos       Suricata                 44112         17585  60.135564                                 713 98.383660
         santos          Wazuh                 82401         21846  73.488186                                 811 99.015789
         santos         AMiner                  4266           358  91.608064                                 211 95.053915
        

In [23]:
statistics

,Dataset,Level,Name,Alerts Before Groups,Groups After,Reduction,Alerts Not Assigned to Any Cluster,Coverage
0,wardbeck,,Suricata,24510,4487,81.693186,16,99.934721
1,wardbeck,,Wazuh,63694,11463,82.003014,357,99.439508
2,wardbeck,,AMiner,3053,552,81.919424,219,92.826728
3,wardbeck,,All IDSs,91257,11812,87.056335,498,99.454288
4,santos,,Suricata,44112,17585,60.135564,713,98.383660
5,santos,,Wazuh,82401,21846,73.488186,811,99.015789
6,santos,,AMiner,4266,358,91.608064,211,95.053915
7,santos,,All IDSs,130779,21991,83.184609,180,99.862363
8,fox,,Suricata,11084,2111,80.954529,17,99.846626
9,fox,,Wazuh,451439,8457,98.126657,283,99.937312


In [24]:
statistics.columns

Index(['Dataset', 'Level', 'Name', 'Alerts Before Groups', 'Groups After',
       'Reduction', 'Alerts Not Assigned to Any Cluster', 'Coverage'],
      dtype='object')

In [28]:
def aggregate_statistics(df):
    """
    Aggregate alert-grouping statistics across all datasets.

    Parameters
    ----------
    df : pandas.DataFrame
        Expected columns:
            Dataset
            Level
            Name
            Alerts Before Groups
            Groups After
            Reduction
            Alerts Not Assigned to Any Cluster
            Coverage

    Returns
    -------
    pandas.DataFrame
        One row each for Suricata, Wazuh, AMiner, and All IDSs.

        Alerts Before and Groups After are summed across datasets.
        Alerts Not Assigned to Any Cluster is summed across datasets.

        Reduction and Coverage are recalculated from the
        aggregated values.
    """

    names = ["Suricata", "Wazuh", "AMiner"]

    results = []

    # ---------------------------------------------------------
    # Aggregate each IDS
    # ---------------------------------------------------------
    for name in names:

        subset = df[df["Name"] == name]

        alerts_before = subset["Alerts Before Groups"].sum()
        groups_after = subset["Groups After"].sum()

        alerts_not_assigned = (
            subset["Alerts Not Assigned to Any Cluster"].sum()
        )

        reduction = (
            (1 - groups_after / alerts_before) * 100
            if alerts_before > 0
            else 0
        )

        coverage = (
            (
                (alerts_before - alerts_not_assigned)
                / alerts_before
            ) * 100
            if alerts_before > 0
            else 0
        )

        results.append({
            "Dataset": name,
            "Alerts Before": alerts_before,
            "Groups After": groups_after,
            "Reduction": reduction,
            "Alerts Not Assigned to Any Cluster": alerts_not_assigned,
            "Coverage": coverage
        })

    # ---------------------------------------------------------
    # Aggregate all three IDSs
    # ---------------------------------------------------------
    ids_df = df[df["Name"].isin(names)]

    alerts_before = ids_df["Alerts Before Groups"].sum()
    groups_after = ids_df["Groups After"].sum()

    alerts_not_assigned = (
        ids_df["Alerts Not Assigned to Any Cluster"].sum()
    )

    reduction = (
        (1 - groups_after / alerts_before) * 100
        if alerts_before > 0
        else 0
    )

    coverage = (
        (
            (alerts_before - alerts_not_assigned)
            / alerts_before
        ) * 100
        if alerts_before > 0
        else 0
    )

    results.append({
        "Dataset": "All IDSs",
        "Alerts Before": alerts_before,
        "Groups After": groups_after,
        "Reduction": reduction,
        "Alerts Not Assigned to Any Cluster": alerts_not_assigned,
        "Coverage": coverage
    })

    return pd.DataFrame(results)


In [29]:
aggregated = aggregate_statistics(statistics)

print(aggregated)

    Dataset  Alerts Before  Groups After  Reduction  \
0  Suricata         306635         84032  72.595431   
1     Wazuh        2293628        132817  94.209305   
2    AMiner          55558          3170  94.294251   
3  All IDSs        2655821        220019  91.715594   

   Alerts Not Assigned to Any Cluster   Coverage  
0                                3739  98.780635  
1                                5163  99.774898  
2                                1751  96.848339  
3                               10653  99.598881  


In [30]:
aggregated

,Dataset,Alerts Before,Groups After,Reduction,Alerts Not Assigned to Any Cluster,Coverage
0,Suricata,306635,84032,72.595431,3739,98.780635
1,Wazuh,2293628,132817,94.209305,5163,99.774898
2,AMiner,55558,3170,94.294251,1751,96.848339
3,All IDSs,2655821,220019,91.715594,10653,99.598881


## Matching the number of alerts specified in the dataset description
According to the official repo of the dataset (https://zenodo.org/records/8263181), the total number of alerts involved in the data set is 2,655,821, of which 2,293,628 origin from Wazuh, 306,635 origin from Suricata, and 55,558 origin from AMiner.